In [1]:
import os
os.chdir("../")

In [2]:
from dataclasses import dataclass
from pathlib import Path
from src.constants import *
from src.utils.common import read_yaml
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_from_disk
import torch

/home/aditya/Desktop/MLOPS/FinancialPhraseClassifier/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass()
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    learning_rate: float
    per_device_train_batch_size: int
    weight_decay: float

In [4]:
config = read_yaml(Path("config/config.yaml"))
params = read_yaml(Path("params.yaml"))
trainer_config = config.model_trainer
training_params = params.TrainingArguments
os.makedirs(trainer_config.root_dir, exist_ok=True)

In [5]:
model_trainer_config = ModelTrainerConfig(
    root_dir=Path(trainer_config.root_dir),
    data_path=Path(trainer_config.data_path),
    model_ckpt=trainer_config.model_ckpt,
    num_train_epochs=training_params.num_train_epochs,
    learning_rate=training_params.learning_rate,
    per_device_train_batch_size=training_params.per_device_train_batch_size,
    weight_decay=training_params.weight_decay
)

In [6]:
dataset = load_from_disk(model_trainer_config.data_path)
tokenizer = AutoTokenizer.from_pretrained(model_trainer_config.model_ckpt)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [7]:
id2label = {0: "bearish", 1: "bullish", 2: "neutral"}
label2id = {"bearish": 0, "bullish": 1, "neutral": 2}

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_trainer_config.model_ckpt,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40155.04it/s]


In [9]:
training_args = TrainingArguments(
    output_dir=model_trainer_config.root_dir,
    learning_rate=model_trainer_config.learning_rate,
    per_device_train_batch_size=2,          
    weight_decay=model_trainer_config.weight_decay,
    max_steps=2,                            
    logging_steps=1,
    save_strategy="no",                     
    use_cpu=True                            
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"]
)

trainer.train()
trainer.save_model(os.path.join(model_trainer_config.root_dir, "finbert-model"))

Step,Training Loss
1,0.307240
2,0.093561


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]
